In [1]:
pip install opencv-python

In [2]:
import os
import time
import copy
import glob
import cv2
import shutil

import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt


In [3]:
data_path='/content/catanddog/train'

transform=transforms.Compose(
    [
        transforms.Resize([256,256]),
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ]
)

train_dataset=torchvision.datasets.ImageFolder(
    data_path, transform=transform
)
train_loader=DataLoader(
    train_dataset, batch_size=32, num_workers=8, shuffle=True
)
print(len(train_dataset))

FileNotFoundError: [Errno 2] No such file or directory: '/content/catanddog/train'

In [ ]:
import numpy as np
samples, labels = next(iter(train_loader))
classes={0:'cat', 1:'dog'}
fig=plt.figure(figsize=(16,24))
for i in range(24):
  a=fig.add_subplot(4,6, i+1)
  a.set_title(classes[labels[i].item()])
  a.axis('off')
  a.imshow(np.transpose(samples[i].numpy(),(1,2,0)))
plt.subplots_adjust(bottom=0.2, top=0.6, hspace=0)

In [ ]:
resnet18=models.resnet18(pretrained=True)

In [ ]:
def set_parameter_requires_grad(model, feature_extracting=True):
  if feature_extracting:
    for param in model.parameters():
      param.requires_grad=False

set_parameter_requires_grad(resnet18)

In [ ]:
resnet18.fc=nn.Linear(512,2)

In [ ]:
for name, param in resnet18.named_parameters():
  if param.requires_grad:
    print(name, param.data)

In [ ]:
model=models.resnet18(pretrained=True)


for param in model.parameters():
  param.requires_grad=False


model.fc=torch.nn.Linear(512,2)
for param in model.fc.parameters():
  param.requires_grad=True

optimizer=torch.optim.Adam(model.fc.parameters())
cost=torch.nn.CrossEntropyLoss()
print(model)

In [ ]:
def train_model(model, dataloaders, criterion, optimizer, device,
                num_epochs=13, is_train=True):

    since = time.time()

    acc_history = []
    loss_history = []
    best_acc = 0.0


    model = model.to(device)

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        model.train()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders:


            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)

            loss = criterion(outputs, labels)

            _, preds = torch.max(outputs, 1)

            loss.backward()
            optimizer.step()


            running_loss += loss.item() * inputs.size(0)

            running_corrects += torch.sum(preds == labels.data)


        epoch_loss = running_loss / len(dataloaders.dataset)

        epoch_acc = (running_corrects.double() / len(dataloaders.dataset))

        print('Loss: {:.4f} Acc: {:.4f}'.format(epoch_loss,epoch_acc))

        if epoch_acc > best_acc:
            best_acc = epoch_acc

        acc_history.append(epoch_acc.item())
        loss_history.append(epoch_loss)


        torch.save(model.state_dict(),os.path.join(
                '/content/catanddog/',
                '{0:0=2d}.pth'.format(epoch)))
        print()

    time_elapsed = time.time() - since

    print('Training complete in {:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))
    print('Best Acc: {:4f}'.format(best_acc))

    return acc_history, loss_history

In [ ]:
params_to_update=[]
for name, param in resnet18.named_parameters():
  if param.requires_grad==True:
    params_to_update.append(param)
    print("\t", name)

optimizer=optim.Adam(params_to_update)

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion=nn.CrossEntropyLoss()
train_acc_hist, train_loss_hist=train_model(resnet18, train_loader, criterion, optimizer, device)

In [ ]:
test_path="/content/catanddog/test"

transform=transforms.Compose(
    [
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
    ]
)

test_dataset=torchvision.datasets.ImageFolder(root=test_path,transform=transform)
test_loader=torch.utils.data.DataLoader(test_dataset, batch_size=32, num_workers=1, shuffle=True)

print(len(test_dataset))

In [ ]:
def eval_model(model, dataloaders, device):
    since = time.time()
    acc_history = []
    best_acc = 0.0

    saved_models = glob.glob('/content/catanddog/' + '*.pth')
    saved_models.sort()

    print('saved_model', saved_models)

    model = model.to(device)

    for model_path in saved_models:
        print('loading model', model_path)

        model.load_state_dict(torch.load(model_path, map_location=device))

        model.eval()

        running_corrects = 0

        for inputs, labels in dataloaders:
            inputs = inputs.to(device)
            labels = labels.to(device)

            with torch.no_grad():
                outputs = model(inputs)

                _, preds = torch.max(outputs, 1)

            running_corrects += torch.sum(preds == labels)

        epoch_acc = (running_corrects.double()/len(dataloaders.dataset))

        print('Acc: {:.4f}'.format(epoch_acc))

        if epoch_acc > best_acc:
            best_acc = epoch_acc

        acc_history.append(epoch_acc.item())

        print()

    time_elapsed = time.time() - since

    print('Validation complete in {:.0f}m {:.0f}s'.format(time_elapsed // 60,time_elapsed % 60))
    print('Best Acc: {:4f}'.format(best_acc))

    return acc_history

In [ ]:
val_acc_hist=eval_model(resnet18, test_loader, device)

In [ ]:
plt.plot(train_acc_hist)
plt.plot(val_acc_hist)
plt.show()

In [ ]:
plt.plot(train_loss_hist)
plt.show()

In [ ]:
def im_convert(tensor):
  image=tensor.clone().detach().numpy()
  image=image.transpose(1,2,0)
  imge=image*(np.array((0.5,0.5,0.5))+np.array((0.5,0.5,0.5)))
  image=image.clip(0,1)
  return image

In [ ]:
classes = {0: 'cat', 1: 'dog'}

dataiter = iter(test_loader)
images, labels = next(dataiter)


images = images.to(device)
labels = labels.to(device)

model = model.to(device)
model.eval()

with torch.no_grad():
    output = model(images)
    _, preds = torch.max(output, 1)

fig = plt.figure(figsize=(25, 4))

for idx in np.arange(20):
    ax = fig.add_subplot(2, 10, idx + 1, xticks=[], yticks=[])


    plt.imshow(im_convert(images[idx].cpu()))

    ax.set_title(
        "{}({})".format(
            classes[preds[idx].item()],
            classes[labels[idx].item()]
        ),
        color=("green" if preds[idx] == labels[idx] else "red")
    )

plt.subplots_adjust(bottom=0.2, top=0.6, hspace=0)
plt.show()

In [ ]:
pip install pillow

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import torch
import torch.nn.functional as F
import torch.nn as nn
from torchvision.transforms import ToTensor
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
class XAI(torch.nn.Module):
    def __init__(self, num_classes=2):
        super(XAI, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),

            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(256, 512, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Conv2d(512, 512, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Conv2d(512, 512, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(512, 512, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Conv2d(512, 512, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Conv2d(512, 512, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.classifier = nn.Sequential(
            nn.Linear(512, 512, bias=False),
            nn.Dropout(0.5),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(-1, 512)
        x = self.classifier(x)
        return F.log_softmax(x)

In [ ]:
model=XAI()
model.to(device)
model.eval()

In [ ]:
class LayerActivations:
    def __init__(self, model, layer_num):
        self.features = None
        self.hook = model[layer_num].register_forward_hook(self.hook_fn)

    def hook_fn(self, module, input, output):
        self.features = output.detach().cpu().numpy()

    def remove(self):
        self.hook.remove()

In [ ]:
img = cv2.imread("/content/cat.jpg")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(img)

img = cv2.resize(img, (100, 100), interpolation=cv2.INTER_LINEAR)
img = ToTensor()(img).unsqueeze(0)

device = next(model.parameters()).device
img = img.to(device)

result = LayerActivations(model.features, 0)

model(img)

activations = result.features

In [ ]:
fig, axes=plt.subplots(4,4)
fig=plt.figure(figsize=(12,8))
fig.subplots_adjust(left=0, right=1, bottom=0, top=1, hspace=0.05, wspace=0.05)
for row in range(4):
  for column in range(4):
    axis=axes[row][column]
    axis.get_xaxis().set_ticks([])
    axis.get_yaxis().set_ticks([])
    axis.imshow(activations[0][row*10+column])
plt.show()

In [ ]:
result=LayerActivations(model.features,20)

model(img)
activations=result.features

In [ ]:
fig, axes=plt.subplots(4,4)
fig=plt.figure(figsize=(12,8))
fig.subplots_adjust(left=0, right=1, bottom=0, top=1, hspace=0.05, wspace=0.05)
for row in range(4):
  for column in range(4):
    axis=axes[row][column]
    axis.get_xaxis().set_ticks([])
    axis.get_yaxis().set_ticks([])
    axis.imshow(activations[0][row*10+column])
plt.show()

In [ ]:
result=LayerActivations(model.features,40)

model(img)
activations=result.features

In [ ]:
fig, axes=plt.subplots(4,4)
fig=plt.figure(figsize=(12,8))
fig.subplots_adjust(left=0, right=1, bottom=0, top=1, hspace=0.05, wspace=0.05)
for row in range(4):
  for column in range(4):
    axis=axes[row][column]
    axis.get_xaxis().set_ticks([])
    axis.get_yaxis().set_ticks([])
    axis.imshow(activations[0][row*10+column])
plt.show()